In [18]:
import pandas as pd
df = pd.read_csv("../datasets/exam_score_baseline.csv")

# Task 1: Define the Modeling Problem

Target variable (y): 
- Exam_Score

Predictors (X):
- Hours_Studied
- Attendance_Percent
- Sleep_Hours
- Assignments_Completed

Synthetic_Record_ID is excluded because it is only an identifier.

In [19]:
numeric_features = [
    "Hours_Studied",
    "Attendance_Percent",
    "Sleep_Hours",
    "Assignments_Completed"
]

X = df[numeric_features]
y = df["Exam_Score"]

df.head()

,Synthetic_Record_ID,Hours_Studied,Attendance_Percent,Sleep_Hours,Assignments_Completed,Exam_Score
0,S001,10.2,93,6.3,4,70
1,S002,11.3,100,7.3,9,77
2,S003,12.6,76,8.0,9,77
3,S004,5.3,84,6.4,7,65
4,S005,5.1,77,5.7,8,52


# Task 2: Predict the Effect of Scale

kNN finds the data points that are closest to each other.

Some predictors have much larger number ranges than others. In our dataset, Attendance_Percent has the largest range, so it could have more influence on the distance calculation just because its numbers are bigger.

Standardizing puts the predictors on a similar scale. This helps kNN compare the predictors more fairly and can change which data points are considered the nearest neighbors.

In [20]:
for feature in numeric_features:
    feature_range = df[feature].max() - df[feature].min()
    print(feature, "range:", feature_range)

Hours_Studied range: 18.0
Attendance_Percent range: 42
Sleep_Hours range: 4.699999999999999
Assignments_Completed range: 9


# Task 3: Prepare the Predictors

All four predictors in the Group 1 baseline dataset are numerical, so no categorical encoding is needed.

The predictors will stay in their original form for now. Scaling will be done later inside the kNN pipeline so that the scaler only uses the training data.

# Task 4: Compare Linear Regression and kNN

This task compares three models:

1. Ordinary Linear Regression
2. kNN with k=5 using the original predictor values
3. kNN with k=5 using standardized predictor values

The goal is to see whether scaling improves kNN performance and how kNN compares with the Linear Regression baseline.

In [21]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Build the Models

The unscaled kNN model uses the original predictor values.

The scaled kNN model uses a pipeline that standardizes the predictors first and then applies kNN.

In [22]:
linear_model = LinearRegression()

knn_unscaled = KNeighborsRegressor(
    n_neighbors=5
)

knn_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=5))
])

## Cross-Validation Setup

All three models are evaluated using the same repeated cross-validation procedure so the comparison is fair.

- 5 folds
- 5 repeats
- 25 total evaluation folds
- random_state = 407

In [23]:
from sklearn.model_selection import RepeatedKFold, cross_validate

cv = RepeatedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=407
)

In [24]:
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

In [25]:
results_linear = cross_validate(
    linear_model,
    X,
    y,
    cv=cv,
    scoring=scoring
)

results_knn_unscaled = cross_validate(
    knn_unscaled,
    X,
    y,
    cv=cv,
    scoring=scoring
)

results_knn_scaled = cross_validate(
    knn_scaled,
    X,
    y,
    cv=cv,
    scoring=scoring
)

## Evaluate the Models

Each model is tested using MAE, RMSE, and R².

- **MAE:** Average prediction error. Lower is better.
- **RMSE:** Similar to MAE, but larger mistakes count more. Lower is better.
- **R²:** Shows how much variation in Exam_Score is explained by the model. Higher is better.

In [26]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "kNN k=5 Unscaled",
        "kNN k=5 Scaled"
    ],
    "MAE Mean": [
        -results_linear["test_MAE"].mean(),
        -results_knn_unscaled["test_MAE"].mean(),
        -results_knn_scaled["test_MAE"].mean()
    ],
    "MAE SD": [
        results_linear["test_MAE"].std(),
        results_knn_unscaled["test_MAE"].std(),
        results_knn_scaled["test_MAE"].std()
    ],
    "RMSE Mean": [
        -results_linear["test_RMSE"].mean(),
        -results_knn_unscaled["test_RMSE"].mean(),
        -results_knn_scaled["test_RMSE"].mean()
    ],
    "RMSE SD": [
        results_linear["test_RMSE"].std(),
        results_knn_unscaled["test_RMSE"].std(),
        results_knn_scaled["test_RMSE"].std()
    ],
    "R2 Mean": [
        results_linear["test_R2"].mean(),
        results_knn_unscaled["test_R2"].mean(),
        results_knn_scaled["test_R2"].mean()
    ],
    "R2 SD": [
        results_linear["test_R2"].std(),
        results_knn_unscaled["test_R2"].std(),
        results_knn_scaled["test_R2"].std()
    ]
})

comparison.round(3)

,Model,MAE Mean,MAE SD,RMSE Mean,RMSE SD,R2 Mean,R2 SD
0,Linear Regression,3.084,0.459,3.766,0.507,0.753,0.065
1,kNN k=5 Unscaled,4.272,0.509,5.135,0.540,0.547,0.088
2,kNN k=5 Scaled,4.394,0.495,5.279,0.509,0.516,0.117


## Results

Linear Regression performed better than both kNN models.

The unscaled kNN model performed slightly better than the scaled kNN model in this comparison.

- Linear Regression had the lowest MAE and RMSE.
- Linear Regression also had the highest R².
- Scaling did not improve kNN performance for k=5 on this baseline dataset.

This shows that scaling can be important for kNN, but it does not guarantee that the scaled model will always perform better.